# Classiq Quantum Circuit Challenge: Depth 117 Champion Verification & Interactive Analysis

This notebook interactively verifies the **depth 117 / 590 CX / 18 qubit champion circuit** for the [Classiq Quantum Circuit Challenge](https://www.classiq.io/challenge).

### Champion Metrics
- **Circuit Depth:** **117** (97.8% reduction from baseline 5,329)
- **CX Gate Count:** **590**
- **U3 Gate Count:** **421**
- **Total Gates:** **1,011**
- **Circuit Width:** **18 qubits** (strict challenge limit)

In [ ]:
import sys, os
from pathlib import Path
# Add src/ to python path
root = Path.cwd().parent if Path.cwd().name == 'jupyter_files' else Path.cwd()
sys.path.insert(0, str(root / 'src'))

import numpy as np
import matplotlib.pyplot as plt
from classiq_synth.core.oracle import logo, get_target_array, MASK
from classiq_synth.core.circuit import load_circuit, get_circuit_metrics
from classiq_synth.core.verify import exhaustive_verify, dense_verify

print('Environment initialized successfully!')

## 1. The Target Oracle: 64x64 Geometric Specification

The oracle computes a diagonal phase shift of -1 if the 12-bit input coordinate (x, y) in [0, 63]^2 falls inside the Classiq logo, and +1 otherwise.

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(MASK, cmap='Blues', origin='lower')
plt.title('Classiq Challenge Target Geometric Logo (64x64)', fontweight='bold')
plt.xlabel('x coordinate (0-63)')
plt.ylabel('y coordinate (0-63)')
plt.colorbar(label='Oracle Phase Flag (1 = Phase pi, 0 = Phase 0)')
plt.show()

## 2. Exhaustive Numerical Verification across All 4,096 Basis States

Using exact sparse-batched unitary simulation, we verify all 2^12 = 4,096 computational basis inputs against the ground-truth geometric predicate.

In [ ]:
qasm_path = root / 'artifacts' / '117' / 'conditional_loader_117.qasm'
report = exhaustive_verify(qasm_path, max_width=18, write_report=False)

import json
print(json.dumps(report, indent=2))

assert report['depth'] == 117
assert report['cx_count'] == 590
assert report['width'] == 18
assert report['max_error'] < 1e-12
assert report['ancilla_error'] < 1e-12
print('\n>>> ALL 4,096 BASIS STATES PASSED EXHAUSTIVE VERIFICATION! <<<')

## 3. Dense Random-State Aer Simulation

We evaluate random quantum state superpositions with full 4,096-dimensional support using Qiskit Aer to confirm zero ancilla leakage and norm preservation.

In [ ]:
dense_report = dense_verify(qasm_path, tests=3, write_report=False)
print(json.dumps(dense_report, indent=2))
print('\n>>> DENSE RANDOM-STATE SIMULATION PASSED! <<<')

## 4. Per-Qubit Wire Profile

Inspect the depth and gate count breakdown across all 18 physical wires.

In [ ]:
wires = [f'q[{i:02d}]' for i in range(18)]
depths = [117, 116, 117, 117, 117, 116, 114, 117, 117, 117, 115, 117, 117, 117, 117, 117, 117, 117]
cx_counts = [86, 79, 85, 83, 58, 60, 83, 76, 64, 58, 60, 72, 74, 50, 60, 52, 44, 36]
u3_counts = [28, 28, 23, 29, 16, 14, 17, 30, 24, 18, 18, 37, 14, 24, 29, 28, 20, 24]

x = np.arange(len(wires))
width = 0.28

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width, depths, width, label='Wire Depth', color='#3b82f6')
ax.bar(x, cx_counts, width, label='CX Count', color='#f97316')
ax.bar(x + width, u3_counts, width, label='U3 Count', color='#10b981')
ax.set_ylabel('Count / Depth')
ax.set_title('Gate & Depth Distribution across 18 Qubits in Champion Circuit', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(wires, rotation=45)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()